In [ ]:
import sys
sys.path.insert(0, '../..')

In [ ]:
%load_ext autoreload
%autoreload 2
import edunn as nn
import numpy as np

# Recurrent Neural Networks

Recurrent neural networks (RNNs) are a family of architectures specifically designed to work with sequential data, such as time series, text or audio. Unlike a traditional dense layer, an **RNN layer has memory**: thanks to its feedback mechanism, it can take into account not only the current input but also everything it has seen before.

They were among the first architectures able to model language contextually, which makes them the precursors of today's large language models (LLMs).

> The files used in this practical guide will be: `edunn\models\cross_entropy_error.py` and `edunn\models\rnn.py`.

In [ ]:
X_train, y_train, word_to_index, index_to_word = nn.datasets.load_sequence_generation("reddit_comments")

In [ ]:
print("x example:", " ".join(index_to_word[x] for x in  X_train[0]))
print("x indices:",  X_train[0])
print("y example:", " ".join(index_to_word[y] for y in y_train[0]))
print("y indices:", y_train[0])

In [ ]:
vocab_size = len(word_to_index)
print(f"Vocabulary size: {vocab_size}")

In [ ]:
index_to_word[0], index_to_word[1], index_to_word[2], index_to_word[3]

In [ ]:
X_train.shape, y_train.shape

In [ ]:
X_train = X_train[:1000]
y_train = y_train[:1000]

In [ ]:
def pad_sequences(sequences, max_len=None, padding_value=0):
    if max_len is None:
        max_len = max(len(seq) for seq in sequences)
    padded = np.full((len(sequences), max_len), padding_value, dtype=np.int32)
    for i, seq in enumerate(sequences):
        padded[i, :len(seq)] = seq[:max_len]
    return padded

In [ ]:
def encode_onehot(sequences, vocab_size, max_len):
    one_hot = np.zeros((len(sequences), max_len, vocab_size))
    for i, seq in enumerate(sequences):
        for t, idx in enumerate(seq):
            if idx < vocab_size:  # Avoid out-of-range indices
                one_hot[i, t, idx] = 1
    return one_hot

In [ ]:
max_len = 50  # Adjust to fit your data
X_train_padded = pad_sequences(X_train, max_len=max_len, padding_value=1)
y_train_padded = pad_sequences(y_train, max_len=max_len, padding_value=1)

In [ ]:
X_train_padded.shape, y_train_padded.shape

In [ ]:
X_train_onehot = encode_onehot(X_train_padded, vocab_size, max_len)
y_train_onehot = encode_onehot(y_train_padded, vocab_size, max_len)

In [ ]:
X_train_onehot.shape, y_train_onehot.shape

In [ ]:
X_train[0], X_train_padded[0], X_train_onehot[0]

## Forward Method

The architecture of an RNN can be described by the triple of parameters $(\mathbf{U} \in \mathbb{R}^{H \times I}, \mathbf{W} \in \mathbb{R}^{H \times H}, \mathbf{V} \in \mathbb{R}^{O \times H})$, which are shared across all time steps.

![alt text](img/rnn_forward.png)

The forward pass equations are:

$$
\begin{aligned}
\mathbf{h}^{(t)} &= \tanh\left(\mathbf{x}^{(t)} \mathbf{U}^\top + \mathbf{h}^{(t-1)} \mathbf{W}^\top\right)
\\
\mathbf{y}^{(t)} &= \mathbf{h}^{(t)} \mathbf{V}^\top
\end{aligned}
$$

where $\mathbf{x}^{(t)} \in \mathbb{R}^{B \times I}$ is the input sequence, $\mathbf{h}^{(t)} \in \mathbb{R}^{B \times H}$ is the **hidden state**, and $\mathbf{y}^{(t)} \in \mathbb{R}^{B \times O}$ is the **output**.

> $B$ is the batch size (`batch_size`), $I$ is the input dimension (`input_dim`), $H$ is the hidden dimension (`hidden_dim`), and $O$ is the output dimension (`output_dim`). We will call $C$ the vocabulary size (`vocab_size`).

There are variants such as [LSTM (Long Short-Term Memory)](https://ieeexplore.ieee.org/abstract/document/6795963) and [GRU (Gated Recurrent Unit)](https://arxiv.org/abs/1406.1078), which were designed to **mitigate the vanishing gradient problem** that traditional RNNs run into when modeling long-range dependencies. Both introduce *gates* that regulate the flow of information through time. LSTM does so with three gates (input, forget, output), while GRU simplifies the architecture by using only two (update and reset).

## Backward Method

![alt text](img/rnn_backward.png)

Knowing that, for a fixed time $t$, the $i$-th component of $\mathbf{y}^{(t)}$ and the gradient of the error function $E$ with respect to that output are given by ($\in\mathbb{R}^{B\times O}$):

$$
y^{(t)}_i = \sum_{k=1}^{H} h^{(t)}_k V_{ik},
\quad
\mathbf{g}^{(t)}\equiv \frac{\partial E}{\partial \mathbf{y}^{(t)}}
$$

> The **Kronecker delta** $\delta_{ki}$ is a compact and elegant way of expressing the condition: the expression only holds if two indices are equal.
>$$
\underbrace{
\frac{\partial}{\partial B_{ij}} \left( \sum_{n=1}^{N} a_n B_{kn} \right)
}_{\mathclap{\text{= } a_j \text{ if and only if } k = i}}
=
\begin{cases}
a_j & \text{if } k = i, \\
0 & \text{if } k \ne i.
\end{cases}
= a_j \cdot \delta_{ki}
>$$

### `δEδV`

$$
\frac{\partial E}{\partial \mathbf{V}} = \sum_{t=0}^{T-1} \frac{\partial E}{\partial \mathbf{y}^{(t)}} \frac{\partial \mathbf{y}^{(t)}}{\partial \mathbf{V}}
$$

Then, by the chain rule:

$$
\frac{\partial E}{\partial V_{ik}}
= \sum_{j=1}^{O} \frac{\partial E}{\partial y^{(t)}_j} \frac{\partial y^{(t)}_j}{\partial V_{ik}}
= \sum_{j=1}^{O} g^{(t)}_j \underbrace{ \frac{\partial}{\partial V_{ik}} \left( \sum_{k=1}^{H} h^{(t)}_k V_{jk} \right) }_{\mathclap{h^{(t)}_k \delta_{ij}}}
= g^{(t)}_i h^{(t)}_k
$$

In other words, each entry of the gradient is an "outer" product between the output error vector and the hidden state, summed over every time step:

$$
\boxed{\quad
\frac{\partial E}{\partial \mathbf{V}}
= \sum_{t=0}^{T-1} \left( \frac{\partial E}{\partial \mathbf{y}^{(t)}} \right)^{\top} \mathbf{h}^{(t)}
= \sum_{t=0}^{T-1} \left( \mathbf{g}^{(t)}\right)^{\top} \mathbf{h}^{(t)} \in\mathbb{R}^{O\times H}
\quad}
$$

<!-- dE_dV += dE_dy[:, t, :].T @ h[:, t, :] -->

### `δEδU`

$$
\frac{\partial E}{\partial \mathbf{U}} = \sum_{t=0}^{T-1} \frac{\partial E}{\partial \mathbf{y}^{(t)}} \frac{\partial \mathbf{y}^{(t)}}{\partial \mathbf{U}}
$$

Then, by the chain rule:

$$
\frac{\partial y^{(t)}_i}{\partial U_{pq}} = \sum_{k=1}^{H} V_{ik} \frac{\partial h^{(t)}_k}{\partial U_{pq}}
$$

We define the "unactivated input":

$$
\mathbf{a}^{(t)} = \mathbf{x}^{(t)} \mathbf{U}^{\top} + \mathbf{h}^{(t-1)} \mathbf{W}^{\top}
\quad \Longrightarrow \quad
a^{(t)}_k = \sum_{m=1}^{I} x^{(t)}_m U_{km} + \sum_{r=1}^{H} h^{(t-1)}_r W_{kr}
$$

Since $h^{(t)}_k = \tanh(a^{(t)}_k)$, and by definition $\tfrac{d\tanh(f(x))}{dx} := \left( 1 - \tanh^{2}(f(x))\right) \cdot \tfrac{df(x)}{dx}$, then:

$$
\frac{\partial h^{(t)}_k}{\partial U_{pq}}
= \left( 1-(h^{(t)}_k)^{2}\right) \frac{\partial a^{(t)}_k}{\partial U_{pq}}
$$

$$
\frac{\partial a^{(t)}_k}{\partial U_{pq}}
= \frac{\partial}{\partial U_{pq}} \left( \sum_{m=1}^{I} x^{(t)}_m U_{km} \right)
= x^{(t)}_q \delta_{kp}
$$

Therefore:

$$
\frac{\partial y^{(t)}_i}{\partial U_{pq}}
= \sum_{k=0}^{H} V_{ik} \left( 1-(h^{(t)}_k)^{2}\right) x^{(t)}_q \delta_{kp}
= V_{ip} \left( 1-(h^{(t)}_p)^{2}\right) x^{(t)}_q
$$

Substituting back into the original expression:

$$
\frac{\partial E}{\partial U_{pq}}
= \sum_{t=0}^{T-1}\sum_{i=1}^{O} \underbrace{\frac{\partial E}{\partial y^{(t)}_i}}_{g^{(t)}_i} \frac{\partial y^{(t)}_i}{\partial U_{pq}}
= \sum_{t=0}^{T-1}\sum_{i=1}^{O} g^{(t)}_i V_{ip} \left( 1-(h^{(t)}_p)^{2}\right) x^{(t)}_q
$$

Now defining the intermediate "errors" (rows):

$$
\underbrace{\delta_h^{(t)}}_{\in\mathbb{R}^{B\times H}} := \mathbf{g}^{(t)}\mathbf{V}
\qquad\text{and}\qquad
\underbrace{\delta_a^{(t)}}_{\in\mathbb{R}^{B\times H}} := \delta_h^{(t)} \odot \left( 1-(\mathbf{h}^{(t)})^{2}\right)
$$

<!-- dE_dh = dE_dy[:, t, :] @ V + dh_next
dh_raw = (1 - h[:, t, :] ** 2) * dE_dh -->

Then $\sum_i g^{(t)}_i V_{ip} = (\delta_h^{(t)})_p$, and

$$
\frac{\partial E}{\partial U_{pq}}
= \sum_{t=0}^{T-1} \underbrace{\left( \delta_h^{(t)} \odot (1-(\mathbf{h}^{(t)})^{2})\right)_p}_{(\delta_a^{(t)})_p} x^{(t)}_q
$$

In compact matrix form (for each $t$):

$$
\boxed{\quad
\frac{\partial E}{\partial \mathbf{U}}
= \sum_{t=0}^{T-1} \left( \frac{\partial E}{\partial \mathbf{y}^{(t)}} \mathbf{V}\right) \odot \left( 1-(\mathbf{h}^{(t)})^{2}\right)^{\top} \mathbf{x}^{(t)}
= \sum_{t=0}^{T-1} \left( \delta_a^{(t)}\right)^{\top} \mathbf{x}^{(t)} \in\mathbb{R}^{H\times I}
\quad}
$$

$\delta_a^{(t)}$ is precisely the "outer product" **(pre-activation error)** × **(current input)**.

<!-- dE_dU += dh_raw.T @ x[:, t, :] -->

### `δEδW`

$$
\frac{\partial E}{\partial \mathbf{W}}
= \sum_{t=0}^{T-1}\frac{\partial E}{\partial \mathbf{y}^{(t)}} \frac{\partial \mathbf{y}^{(t)}}{\partial \mathbf{W}}
$$

Then, by the chain rule:

$$
\frac{\partial y^{(t)}_i}{\partial W_{pr}}
= \sum_{k=0}^{H} V_{ik} \frac{\partial h^{(t)}_k}{\partial W_{pr}}
$$

Using the same definition of $\mathbf{a}^{(t)}$ (through $\tanh$) and holding $\mathbf{h}^{(t-1)}$ constant in the local Jacobian:

$$
\frac{\partial h^{(t)}_k}{\partial W_{pr}}
= \left( 1-(h^{(t)}_k)^{2}\right) \frac{\partial a^{(t)}_k}{\partial W_{pr}}
$$

$$
\frac{\partial a^{(t)}_k}{\partial W_{pr}}
= \frac{\partial}{\partial W_{pr}} \left( \sum_{l=1}^{H} h^{(t-1)}_l W_{kl} \right)
= h^{(t-1)}_{r} \delta_{kp}
$$

Therefore:

$$
\frac{\partial y^{(t)}_i}{\partial W_{pr}}
= \sum_k V_{ik} \left( 1-(h^{(t)}_k)^{2}\right) h^{(t-1)}_{r} \delta_{kp}
= V_{ip} \left( 1-(h^{(t)}_p)^{2}\right) h^{(t-1)}_{r}
$$

Substituting back into the original expression:

$$
\frac{\partial E}{\partial W_{pr}}
= \sum_{t=0}^{T-1}\sum_{i=1}^{O} \underbrace{\frac{\partial E}{\partial y^{(t)}_i}}_{g^{(t)}_i} \frac{\partial y^{(t)}_i}{\partial W_{pr}}
= \sum_{t=0}^{T-1}\sum_{i=1}^{O} g^{(t)}_i V_{ip} \left( 1-(h^{(t)}_p)^{2}\right) h^{(t-1)}_{r}
$$

Using the same definitions of $\delta_h^{(t)}$ and $\delta_a^{(t)}$ as before, we again have $\sum_i g^{(t)}_i V_{ip} = (\delta_h^{(t)})_p$ and:

$$
\boxed{\quad
\frac{\partial E}{\partial \mathbf{W}}
= \sum_{t=0}^{T-1} \left( \frac{\partial E}{\partial \mathbf{y}^{(t)}} \mathbf{V}\right) \odot \left( 1-(\mathbf{h}^{(t)})^{2}\right)^{\top} \mathbf{h}^{(t-1)}
= \sum_{t=0}^{T-1} \left( \delta_a^{(t)}\right)^{\top} \mathbf{h}^{(t-1)} \in\mathbb{R}^{H\times H}
\quad}
$$

(at $t=0$ the term is zero because $ \mathbf{h}^{(-1)} = \mathbf{0}$: updating the weights of $\mathbf{W}$ depends on the previous state, and since there is no previous hidden state at the first time step, there is no contribution to the gradient of $\mathbf{W}$ at $t=0$).

<!-- if t > 0:
    dE_dW += dh_raw.T @ h[:, t-1, :] -->

### `δEδx`

$$
\frac{\partial E}{\partial \mathbf{x}^{(t)}}
= \frac{\partial E}{\partial \mathbf{y}^{(t)}} \frac{\partial \mathbf{y}^{(t)}}{\partial \mathbf{x}^{(t)}}
$$

Then, by the chain rule:

$$
\frac{\partial y^{(t)}_i}{\partial x^{(t)}_q}
= \sum_k V_{ik} \frac{\partial h^{(t)}_k}{\partial x^{(t)}_q}
$$

Using the same definition of $\mathbf{a}^{(t)}$ (through $\tanh$):

$$
\frac{\partial h^{(t)}_k}{\partial x^{(t)}_q}
= \left( 1-(h^{(t)}_k)^{2}\right) \frac{\partial a^{(t)}_k}{\partial x^{(t)}_q}
$$

$$
\frac{\partial a^{(t)}_k}{\partial x^{(t)}_q}
= \frac{\partial}{\partial x^{(t)}_q} \left( \sum_{m=1}^{I} x^{(t)}_m U_{km} \right)
= U_{kq}
$$

Therefore:

$$
\frac{\partial y^{(t)}_i}{\partial x^{(t)}_q}
= \sum_{k=1}^{H} V_{ik} \left( 1-(h^{(t)}_k)^{2}\right) U_{kq}
$$

Substituting back into the original expression:

$$
\frac{\partial E}{\partial x^{(t)}_q}
= \sum_{i=1}^{O} \underbrace{\frac{\partial E}{\partial y^{(t)}_i}}_{g^{(t)}_i} \frac{\partial y^{(t)}_i}{\partial x^{(t)}_q}
= \sum_{i=1}^{O} \sum_{k=1}^{H} g^{(t)}_i V_{ik} \left( 1-(h^{(t)}_k)^{2}\right) U_{kq}
$$

Using the same definitions of $\delta_h^{(t)}$ and $\delta_a^{(t)}$ as before, we again have $\sum_i g^{(t)}_i V_{ip} = (\delta_h^{(t)})_p$ and:

$$
\boxed{\quad
\frac{\partial E}{\partial \mathbf{x}^{(t)}}
= \left( \frac{\partial E}{\partial \mathbf{y}^{(t)}} \mathbf{V}\right) \odot \left( 1-(\mathbf{h}^{(t)})^{2}\right) \mathbf{U}
= \delta_a^{(t)} \mathbf{U} \in\mathbb{R}^{B\times I}
\quad}
$$

<!-- dE_dx[:, t, :] = dh_raw @ U -->

### `δEδh`

So far the computations have only covered the **direct** contribution, but for a dependency through time to exist we must automatically account for the paths by which $\mathbf{U}$ and $\mathbf{W}$ affect $\mathbf{h}^{(t-1)},\mathbf{h}^{(t-2)},\ldots$, accumulating gradients (which is exactly what BPTT is).

Within the total error $E = \sum_s E^{(s)}$, $\mathbf{x}^{(t)}$ also affects future outputs ($s>t$) via $\mathbf{h}^{(t)} \to \mathbf{h}^{(t+1)} \to \cdots$. In practice this ends up folded into $\delta_a^{(t)}$, because it is computed with the backward recurrence (which adds in the "future flow").

To the term $\delta_h^{(t)}$ we must add $\boldsymbol{\phi}^{(t)}$, which is the **future** contribution (from the previous state) of the gradient on $\mathbf{h}^{(t)}$, so that the term $\delta_a^{(t)}$ is affected by it.

$$
\frac{\partial E}{\partial \mathbf{h}^{(t)}}
= \underbrace{\frac{\partial E^{(t)}}{\partial \mathbf{h}^{(t)}}}_{\text{local contribution}}
+ \underbrace{\sum_{s>t}\frac{\partial E^{(s)}}{\partial \mathbf{h}^{(t)}}}_{\text{future contribution}}
$$

<center>

| Local contribution (via $\mathbf{y}^{(t)}$) | Future contribution (via $\mathbf{h}^{(t)} \to \mathbf{h}^{(t+1)} \to \cdots$), whose first link is $t \to t+1$: |
|:---------:|:---------:|
| $$ \frac{\partial E^{(t)}}{\partial \mathbf{h}^{(t)}} = \frac{\partial E^{(t)}}{\partial \mathbf{y}^{(t)}} \frac{\partial \mathbf{y}^{(t)}}{\partial \mathbf{h}^{(t)}} = \mathbf{g}^{(t)} \mathbf{V} $$ | $$ \sum_{s>t}\frac{\partial E^{(s)}}{\partial \mathbf{h}^{(t)}} = \frac{\partial E}{\partial \mathbf{h}^{(t+1)}} \frac{\partial \mathbf{h}^{(t+1)}}{\partial \mathbf{h}^{(t)}} = \frac{\partial E}{\partial \mathbf{h}^{(t+1)}} \underbrace{\frac{\partial \mathbf{h}^{(t+1)}}{\partial \mathbf{a}^{(t+1)}}}_{\text{diag}\left( 1-(\mathbf{h}^{(t+1)})^{2}\right)} \underbrace{\frac{\partial \mathbf{a}^{(t+1)}}{\partial \mathbf{h}^{(t)}}}_{\mathbf{W}} = \underbrace{\left( \frac{\partial E}{\partial \mathbf{h}^{(t+1)}} \odot (1-(\mathbf{h}^{(t+1)})^{2})\right)}_{\boldsymbol{\delta}_a^{(t+1)}} \mathbf{W} $$ |

</center>

<!-- #### Local contribution (via $\mathbf{y}^{(t)}$)

$$
\frac{\partial E^{(t)}}{\partial \mathbf{h}^{(t)}}
= \frac{\partial E^{(t)}}{\partial \mathbf{y}^{(t)}}\frac{\partial \mathbf{y}^{(t)}}{\partial \mathbf{h}^{(t)}}
= \mathbf{g}^{(t)} \mathbf{V}
$$

#### Future contribution (via $\mathbf{h}^{(t)} \to \mathbf{h}^{(t+1)} \to \cdots$)

The first link is $t \to t+1$:

$$
\sum_{s>t}\frac{\partial E^{(s)}}{\partial \mathbf{h}^{(t)}}
= \frac{\partial E}{\partial \mathbf{h}^{(t+1)}} \frac{\partial \mathbf{h}^{(t+1)}}{\partial \mathbf{h}^{(t)}}
= \frac{\partial E}{\partial \mathbf{h}^{(t+1)}}
\underbrace{\frac{\partial \mathbf{h}^{(t+1)}}{\partial \mathbf{a}^{(t+1)}}}_{\text{diag}\left( 1-(\mathbf{h}^{(t+1)})^{2}\right)}
\underbrace{\frac{\partial \mathbf{a}^{(t+1)}}{\partial \mathbf{h}^{(t)}}}_{\mathbf{W}}
= \underbrace{\left( \frac{\partial E}{\partial \mathbf{h}^{(t+1)}} \odot (1-(\mathbf{h}^{(t+1)})^{2})\right)}_{\boldsymbol{\delta}_a^{(t+1)}} \mathbf{W}
$$

#### Final recurrence that injects $\boldsymbol{\phi}^{(t)}$ -->

The final recurrence that injects $\boldsymbol{\phi}^{(t)}$ is:

$$
\boxed{\quad
\frac{\partial E}{\partial \mathbf{h}^{(t)}}
= \mathbf{g}^{(t)} \mathbf{V} + \underbrace{\boldsymbol{\delta}_a^{(t+1)} \mathbf{W}}_{\boldsymbol{\phi}^{(t)}} \in\mathbb{R}^{B\times H}
\quad}
$$

where we define $\delta_h^{(t)} \equiv \tfrac{\partial E}{\partial \mathbf{h}^{(t)}}$. At the boundary $t=T-1$ there is no future, so the flow $\boldsymbol{\phi}^{(t)} = 0$, and there is no previous state to propagate the gradient to.

<!-- if t > 0:
    dh_next = dh_raw @ W
else:
    dh_next = 0  # There is no previous state for t=0 -->


## Implementation

In an RNN where the time steps are not vectorized, the sequence has to be traversed one step at a time, so **`forward`** uses an **ascending** loop: each new hidden state $\mathbf{h}^{(t)}$ depends on the previous state $\mathbf{h}^{(t-1)}$, and since information flows from left to right, that is, we generate $\mathbf{h}^{(0)}, \mathbf{h}^{(1)}, \mathbf{h}^{(2)},\dots$ in sequence, the range of the for loop must follow chronological order: `for t in range(timesteps)`. 

```python
for t in range(timesteps):
    h_t = tanh(x_t·Uᵀ + h_(t−1)·Wᵀ)
    y_t = h_t·Vᵀ
    h_(t-1) = h_t
```

In **backpropagation through time (BPTT)**, by contrast, the derivatives of every step with respect to the shared parameters `(U, W, V)` must be accumulated and the error propagated backwards along the sequence: the gradient at $\mathbf{h}^{(t-1)}$ receives contributions both from its own loss at step $t-1$ and from the feedback (through $\mathbf{W}$) of the correction at $\mathbf{h}^{(t)}$. To honor this dependency, **`backward`** must start at the last step $T$ and work **downwards** to $0$, accumulating the gradient "backwards" in time.

```python
for t in reversed(range(timesteps)):
    # gradients of V, U, W, dx and propagation of dh
```

## Initialization

Initializing the `(U, W, V)` parameters is remarkably important for this architecture. They cannot all be initialized to the same value, since that would result in a symmetric computation across every layer. Several works, such as [Xavier Glorot](https://proceedings.mlr.press/v9/glorot10a)'s, have shown how much a correct parameter initialization before training affects the results. Here the weights must be initialized randomly, and since the right initialization depends on the `tanh` activation function, the recommended approach is to initialize them with the `XavierUniform` initializer.

RNNs also suffer from a problem called the *vanishing gradient problem*, which happens when gradients shrink toward 0. This is why traditional RNNs cannot capture long-range dependencies, so we use the `bptt_truncate` parameter to limit the length of the dependency.


In [ ]:
from edunn.initializers import XavierUniform
hidden_dim = 128
model = nn.RNN(vocab_size, vocab_size, hidden_dim, kernel_initializer=XavierUniform())

## Sequence Cross Entropy

At every time step $t$, the output $\mathbf{y}^{(t)}$ can be passed through a $\text{softmax}$, which lets us use the **cross-entropy** loss together with **gradient descent** optimization or a similar method, as long as we keep a few things in mind.

$$E_{t}(\mathbf{y}^{(t)},\hat{\mathbf{y}}^{(t)}) =-\mathbf{y}^{(t)}\log\hat{\mathbf{y}}^{(t)} \quad \Rightarrow \quad E(\mathbf{y},\hat{\mathbf{y}})=\sum^{T-1}_{t=0}E_{t}(\mathbf{y}^{(t)},\hat{\mathbf{y}}^{(t)}) = -\sum^{T-1}_{t=0}\mathbf{y}^{(t)}\log\hat{\mathbf{y}}^{(t)}$$

Here, $\mathbf{y}^{(t)}$ is the correct word at time step $t$, and $\hat{\mathbf{y}}^{(t)}$ is our prediction. We normally treat the whole sequence (the sentence) as a single training example, so the total error is simply the sum of the errors at each time step (each word).

### Forward Method

The RNN returns a vector $\hat{\mathbf{y}}$ of logits with shape `(batch_size, timesteps, vocab_size)`. In order to use the cross-entropy loss function, those logits must go through a $\text{softmax}$ that turns them into a probability distribution, so they can be compared against the correct label $\mathbf{y}$ (in one-hot representation).

$$
E_{t}\bigl(\mathbf{y}^{(t)},\,\hat{\mathbf{y}}^{(t)}\bigr)
\;=\;
-\sum_{c=1}^{C}y^{(t)}_{c}\,\log \bigl(\text{softmax}(\hat{\mathbf{y}}^{(t)})_{c}\bigr)

\quad
\Rightarrow
\quad

E(\mathbf{y},\hat{\mathbf{y}})
\;=\;
\sum_{t=0}^{T-1}E_{t}\bigl(\mathbf{y}^{(t)},\,\hat{\mathbf{y}}^{(t)}\bigr)
\;=\;
-\,\sum_{t=0}^{T-1}\sum_{c=1}^{C}y^{(t)}_{c}\,\log \bigl(\text{softmax}(\hat{\mathbf{y}}^{(t)})_{c}\bigr)
$$

> NOTE: padding tokens (the ones that merely fill the sequence up to the fixed length) must be ignored, and the result normalized over the number of valid tokens.

#### Implementation

We define the **valid-token mask** as $\mathrm{mask}_{b,t} = \mathbb{1}_{\mathrm{token}_{b,t} \neq \text{padding}} \in \{0,1\}^{B\times T}$. Next we turn the logit vector $\hat{\mathbf{y}}^{(b,t)}\in\mathbb{R}^{C}$ into probabilities $\mathbf{p}^{(b,t)} = \text{softmax}(\hat{\mathbf{y}}^{(b,t)})\in[0,1]^{C}$. We then compute $E_{b,t} = - \sum_{c} \mathbf{y}^{(b,t)}_{c} \log p^{(b,t)}_{c}$ to pick out the **true probabilities**, since $c$ is the correct token at $(b,t)$.

Finally, we sum over all $(b,t)$, ignoring padding by multiplying by the mask, and divide by the sum of the mask, since that quotient is a per-token mean, which keeps the magnitude of the loss comparable no matter how many time steps each batch has.

$$
\boxed{\quad
E =  \frac{1}{\sum_{b,t}\mathrm{mask}_{b,t}} \sum_{b=1}^{B}\sum_{t=0}^{T-1} E_{b,t}                                                  \mathrm{mask}_{b,t}
  = -\frac{1}{\sum_{b,t}\mathrm{mask}_{b,t}} \sum_{b=1}^{B}\sum_{t=0}^{T-1} \sum_{c=1}^{C} \mathbf{y}^{(b,t)}_{c} \log p^{(b,t)}_{c} \mathrm{mask}_{b,t}
\quad}
$$

> The function $\text{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$ can produce very small or very large values when the $z_i$ (*logits*) have large or widely differing magnitudes. That can lead to numerical **overflow** or **underflow** during the computation. By contrast, $\log\text{softmax}(\mathbf{z})_i = z_i - \log\sum_j e^{z_j}$ lets us subtract the log-sum-exp **before** exponentiating, which stabilizes the computation and avoids numerical errors.
>
> On top of that, **for numerical stability**, the maximum is subtracted inside `log_softmax`, because the critical part $\sum_j e^{z_j}$ is fragile: if some $z_j$ is very large, $e^{z_j}$ evaluates to `inf`:
>
> $$
> \textstyle
> \log\text{softmax}(\mathbf{z})_i = z_i - \log\sum_j e^{z_j}
> \quad \longrightarrow \quad
> z_i - z_{\text{max}} - \log\sum_j e^{z_j - z_{\text{max}}}
> $$

### Backward Method

The derivative of cross-entropy combined with softmax, for a logit vector $\hat{\mathbf{y}}^{(b,t)}\in\mathbb{R}^{C}$ and a one-hot label vector $\mathbf{y}^{(b,t)}\in\{0,1\}^{C}$, is computed as:

$$
\frac{\partial E}{\partial \hat{\mathbf{y}}^{(b,t)}} = \frac{\partial E}{\partial \mathbf{p}^{(b,t)}} \frac{\partial \mathbf{p}^{(b,t)}}{\partial \hat{\mathbf{y}}^{(b,t)}}
$$

> We use the fact that $\dfrac{\partial}{\partial \hat z_j}\log p_i = \frac{1}{p_i}\frac{\partial p_i}{\partial \hat z_j}$ and that the softmax Jacobian is $\dfrac{\partial p_i}{\partial z_j} = p_i (\delta_{ij} - p_j)$.


### `dp_dŷ`

Let us fix a pair $(b,t)$ and differentiate with respect to the logit $\hat y^{(b,t)}_j$:

$$
\frac{\partial p^{(b,t)}_c}{\partial \hat y^{(b,t)}_j}
= p^{(b,t)}_c (\delta_{cj} - p^{(b,t)}_j)
$$

### `dE_dp`

Let us fix a pair $(b,t)$ and differentiate with respect to the logit $\hat y^{(b,t)}_j$:

$$
\frac{\partial}{\partial \hat y^{(b,t)}_j}\big(\log p^{(b,t)}_c\big)
= \frac{1}{p^{(b,t)}_c} \cdot p^{(b,t)}_c (\delta_{cj} - p^{(b,t)}_j) = \delta_{cj} - p^{(b,t)}_j
$$

and therefore

$$
\frac{\partial E}{\partial \hat y^{(b,t)}_j}
= -\frac{\mathrm{mask}_{b,t}}{\sum_{b',t'}\mathrm{mask}_{b',t'}} \sum_{c=1}^C y^{(b,t)}_c \frac{\partial}{\partial \hat y^{(b,t)}_j}\big(\log p^{(b,t)}_c\big)
= -\frac{\mathrm{mask}_{b,t}}{\sum_{b',t'}\mathrm{mask}_{b',t'}} \sum_{c=1}^C y^{(b,t)}_c (\delta_{cj}-p^{(b,t)}_j)
= -\frac{\mathrm{mask}_{b,t}}{\sum_{b',t'}\mathrm{mask}_{b',t'}} \big(y^{(b,t)}_j - p^{(b,t)}_j\big).
$$

Rearranging the signs and writing it in vector form for that $(b,t)$:

$$
\boxed{\quad
\frac{\partial E}{\partial \hat{\mathbf y}^{(b,t)}}
= \frac{\mathrm{mask}_{b,t}}{\sum_{b',t'}\mathrm{mask}_{b',t'}} \big(\mathbf p^{(b,t)} - \mathbf y^{(b,t)}\big)
\quad}
$$


#### Implementation

We rebuild the probabilities $\mathbf{p}^{(b,t)}\in[0,1]^{C}$ and define the initial gradient in the space $\mathbb{R}^{B\times T\times C}$ as $g^{(b,t)}_{c} = p^{(b,t)}_{c} - y^{(b,t)}_{c}$; finally we apply the mask and normalize again at the level of the sequence tokens, just as we did in the forward pass.

$$
\frac{\partial E}{\partial \hat{\mathbf{y}}^{(b,t)}} = \mathbf{g}^{(b,t)} \odot \frac{\mathrm{mask}_{b,t}}{\sum_{b',t'}\mathrm{mask}_{b',t'}}
$$

and across the whole batch/time dimension the same expression applies element-wise (broadcasting the mask with `mask[:,:,None]`).

In [ ]:
error = nn.SequenceCrossEntropyWithLabels(pad_index=1)
optimizer = nn.GradientDescent(lr=0.1, epochs=100, batch_size=16)

In [ ]:
# history = optimizer.optimize(model, X_train_onehot, y_train_onehot, error)
# nn.plot.plot_history(history, error_name=error.name)

In [ ]:
y_pred = model.forward(X_train_onehot[:1])
y_pred_labels = nn.utils.onehot2labels(y_pred)

In [ ]:
y_pred.shape, y_pred_labels.shape

In [ ]:
decode_greedy = lambda seq, eos=2: list(seq)[:list(seq).index(eos)] if eos in seq else list(seq)

In [ ]:
print("X:", " ".join(index_to_word[x] for x in X_train[0]))
print("y:", " ".join(index_to_word[y] for y in y_train[0]))
print("ŷ:", " ".join(index_to_word[y] for y in decode_greedy(y_pred_labels[0])))

# Copy Task

The previous dataset, `reddit_comments`, is similar to a classic binary classification task known as the "copy-task" (or, more informally, the "echo-task"). This task is ideal for checking the temporal memory logic through one-hot encoded outputs. However, since that dataset has `vocab_size=8000`, training an RNN on it takes many epochs and a considerable amount of data.

The task consists of presenting a sequence followed by a period of *silence* (padding), and then asking the model to **reproduce** the original sequence.

To make training easier to control, we can automatically generate a dataset in which `X[t] == X[t-k]` holds, that is, the model has to predict the symbol that occurred `k` steps earlier. This integer sequence-to-sequence task lets us verify that the RNN learns temporal dependencies, since it forces the network to **memorize** or **retain** information over multiple steps. That way we can confirm that the hidden state and the backpropagation through time (BPTT) algorithm are working correctly.

> If your network **learns to repeat** the correct token with a low loss, it means BPTT, state handling and gradient computation are implemented correctly.  
> If it does not, you will know the fault lies in the recurrent part or in the propagation through time, and not in the complexity of natural language.


In [ ]:
def generate_echo_dataset(vocab_size: int,
                          seq_len: int,
                          delay: int,
                          num_sequences: int):
    """
    Generates a toy 'echo' dataset.

    Args:
        vocab_size (int): Size of the vocabulary, including PAD token at index 0.
        seq_len (int): Length of each sequence.
        delay (int): How many steps to delay the echo (target output).
        num_sequences (int): Number of sequences to generate.

    Returns:
        tuple[np.ndarray, np.ndarray]: A pair of arrays (X_onehot, Y_onehot),
        both with shape (num_sequences, seq_len, vocab_size).
    """

    X_int = np.random.randint(1, vocab_size, size=(num_sequences, seq_len))
    Y_int = np.zeros_like(X_int) # the first `delay` ones are PAD
    Y_int[:, delay:] = X_int[:, :-delay]

    X_onehot = np.eye(vocab_size)[X_int]
    Y_onehot = np.eye(vocab_size)[Y_int]
    
    return X_onehot, Y_onehot

In [ ]:
vocab_size   = 51    # 0=PAD, 1..50 actual symbols
seq_len      = 20
delay        = 3
num_sequences = 2048

X_echo, Y_echo = generate_echo_dataset(vocab_size, seq_len, delay, num_sequences)

print("X_int example:", np.argmax(X_echo[0], axis=1))
print("Y_int example:", np.argmax(Y_echo[0], axis=1))

In [ ]:
hidden_dim = 16
model = nn.RNN(input_dim=vocab_size,
               output_dim=vocab_size,
               hidden_dim=hidden_dim,
               bptt_truncate=seq_len,
               kernel_initializer=XavierUniform())

error     = nn.SequenceCrossEntropyWithLabels(pad_index=0)
optimizer = nn.GradientDescent(lr=0.1, epochs=200, batch_size=32)

In [ ]:
history = optimizer.optimize(model, X_echo, Y_echo, error)
nn.plot.plot_history(history, error_name=error.name)

In [ ]:
y_pred_logits = model.forward(X_echo[:1])
y_pred_idx    = nn.utils.onehot2labels(y_pred_logits)[0]

In [ ]:
X_echo[:1].shape, y_pred_logits.shape

In [ ]:
print("X:", np.argmax(X_echo[0], axis=1))
print("y:", np.argmax(Y_echo[0], axis=1))
print("ŷ:", np.argmax(y_pred_logits[0], axis=1))

In [ ]:
# define the initial state (t = -1) -> h_prev
# Forward pass through time (for i to timesteps)
    # Compute the hidden state: h_t = tanh(x_t @ U.T + h_prev @ W.T)
    # Compute the output: y_t = h_t @ V.T
    # Store the state and update it for the next timestep

# Backward pass through time (in reverse)
    # # Gradient for V: dE/dV += (dE_dy_t.T @ h_t)
    # dE_dV += np.dot(
    #     dE_dy[:, t, :].T, hs[:, t, :]  # (input_dim, batch_size)  # (batch_size, hidden_dim)
    # )  # Result: (input_dim, hidden_dim)

    # # Gradient of the hidden state (from the output and the next state)
    # dE_dh = np.dot(dE_dy[:, t, :], V) + dh_next  # (batch_size, hidden_dim)  # (batch_size, hidden_dim)

    # # Gradient through tanh (dhraw = dE_dh * (1 - hs[t]^2))
    # dh_raw = (1 - hs[:, t, :] ** 2) * dE_dh

    # # Gradient for U: dE/dU += (dh_raw.T @ x_t)
    # dE_dU += np.dot(
    #     dh_raw.T, x[:, t, :]  # (hidden_dim, batch_size)  # (batch_size, input_dim)
    # )  # Result: (hidden_dim, input_dim)

    # # Gradient for the input x_t: dE/dx_t = dh_raw @ U
    # dE_dx[:, t, :] = np.dot(dh_raw, U)  # (batch_size, input_dim)

    # # Update dh_next if this is not the first timestep
    # if t > 0:
    #     # Gradient for W: dE/dW += (dh_raw.T @ h_{t-1})
    #     dE_dW += np.dot(
    #         dh_raw.T, hs[:, t - 1, :]  # (hidden_dim, batch_size)  # (batch_size, hidden_dim)
    #     )  # Result: (hidden_dim, hidden_dim)

    #     # Prepare the gradient for the next state: dh_next = dh_raw @ W
    #     dh_next = np.dot(dh_raw, W)  # (batch_size, hidden_dim)
    # else:
    #     # At t=0 there is no previous state
    #     dh_next = np.zeros((batch_size, self.hidden_dim))